# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 Dataset Package](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"URL: {croissant_url}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets and their fields using their `@id` fields.

Let's list all record sets present in the dataset with their `@id` and `name` if available.

In [ ]:
# List all record sets in the dataset, display their @id and basic field info
record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record set(s)\n")
for rset in record_sets:
    print(f"RecordSet @id: {rset.id}")
    print(f"  Name: {getattr(rset, 'name', None)}")
    print(f"  Description: {getattr(rset, 'description', None)}")
    # List all field ids for this record set
    if hasattr(rset, 'fields'):
        print("  Fields (@id):")
        for f in rset.fields:
            print(f"    - {f.id}")
    print('-' * 40)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll select the primary record set (the first one listed) and extract a few records to get a sense of the data.

In [ ]:
# If at least one record set exists, extract records into a DataFrame
if not record_sets:
    raise RuntimeError("No record sets defined in the dataset metadata.")

# Select the first record set @id
main_record_set = record_sets[0]
record_set_id = main_record_set.id  # This is the @id to use below
print(f"Using RecordSet: {record_set_id}")

# List data records
example_records = list(dataset.records(record_set=record_set_id))
print(f"Loaded {len(example_records)} records (showing up to 3 examples):")
for rec in example_records[:3]:
    print(rec)

# Load into DataFrame
df = pd.DataFrame(example_records)
print("\nDataFrame columns (field `@id`s):")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform some simple analysis and processing steps. We'll select a numeric field from those present,
and demonstrate basic filtering, normalization, and grouping. All field references will use `@id` as column names (as per Croissant).

In [ ]:
# Display fields for reference
print("Fields in record set (with @id):")
for field in main_record_set.fields:
    print(f"- {field.id} (type: {getattr(field, 'data_type', None)})")

# Pick a numeric field (e.g., sequence_coverage or mw_kda if present)
possible_numeric_fields = [f.id for f in main_record_set.fields if (hasattr(f, 'data_type') and str(f.data_type).lower() in ('float', 'number', 'integer'))]
print(f"Possible numeric fields: {possible_numeric_fields}")
if not possible_numeric_fields:
    raise RuntimeError("No numeric fields found in this record set.")
numeric_field = possible_numeric_fields[0]

# Remove non-numeric data and cast column
df_numeric = df.copy()
df_numeric[numeric_field] = pd.to_numeric(df_numeric[numeric_field], errors='coerce')
df_numeric_clean = df_numeric.dropna(subset=[numeric_field])

threshold = df_numeric_clean[numeric_field].quantile(0.9)  # Top 10% as example filter
filtered_df = df_numeric_clean[df_numeric_clean[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (top decile):")
print(filtered_df[[numeric_field]].head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Optional: Try grouping by a categorical field (e.g., 'accession' or any non-numeric field)
possible_group_fields = [f.id for f in main_record_set.fields if f.id != numeric_field]
group_field = None
for field in possible_group_fields:
    if field in df.columns and df[field].nunique() > 1:
        group_field = field
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nMean {numeric_field} grouped by {group_field} (showing 5):")
    print(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
We can visualize the distribution of the selected numeric field and its normalized version. Here we'll use matplotlib for a histogram and a boxplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_numeric_clean[numeric_field], kde=True, ax=axes[0])
axes[0].set_title(f"Distribution of {numeric_field}")
sns.boxplot(x=df_numeric_clean[numeric_field], ax=axes[1])
axes[1].set_title(f"Boxplot of {numeric_field}")
plt.tight_layout()
plt.show()

# Visualize normalized after filtering
plt.figure(figsize=(6,4))
sns.histplot(filtered_df[f'{numeric_field}_normalized'], bins=10, kde=True)
plt.title(f"Filtered & Normalized {numeric_field}")
plt.xlabel(f"{numeric_field} (normalized)")
plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to explore and process the FAIR^2 human protein mass spectrometry dataset using the `mlcroissant` library, referencing all fields and record sets by their `@id` for reproducibility. Simple EDA steps included filtering high-value records, normalization, grouping by categorical attributes, and basic visualizations. For further analysis, you could explore additional fields and record sets, or use domain-specific techniques for proteomics data.